# AT-FedIDS: Reference Implementation and Experiment Verification

**Adaptive Transformer-based Federated Intrusion Detection with Per-Device-Tier
Differential Privacy for the IoT.**

A self-contained, **config-driven** implementation of AT-FedIDS that runs every
experiment end-to-end on **real intrusion-detection datasets** (CICIDS-2017,
TON-IoT, UNSW-NB15) — or a synthetic fallback. Select any subset of datasets in
the config and the whole pipeline (E1–E7) runs for each.

Implemented & verified:
1. Linear-attention **transformer** local model + **CNN-LSTM** baseline.
2. **DP-SGD** (per-sample clipping + Gaussian noise) and a dependency-free
   **Rényi-DP accountant** with **per-device-tier** noise calibration.
3. **Personalised FedProx** (shared encoder aggregated, head kept local +
   local head adaptation) and FedAvg / FedProx baselines.
4. **Non-IID** (Dirichlet) partitioning across device tiers.
5. Evaluation: per-client accuracy/precision/recall/F1/FPR/AUC, **per-tier
   privacy–utility**, **MIA** resistance, **non-IID convergence**, **ablation**,
   and **latency / communication**.

## §0 — Setup and Imports

In [1]:
from __future__ import annotations
import os, time, math, copy, random, json, warnings
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
try:
    from torch.func import functional_call, vmap, grad as func_grad
    _HAS_FUNC = True
except Exception:
    _HAS_FUNC = False

import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, roc_auc_score)

warnings.filterwarnings("ignore")
try:
    get_ipython().run_line_magic("matplotlib", "inline")  # type: ignore # noqa
    _IN_NOTEBOOK = True
except Exception:
    matplotlib.use("Agg")
    _IN_NOTEBOOK = False

OUTDIR = "nb_outputs"
os.makedirs(OUTDIR, exist_ok=True)


def pick_device():
    """Prefer CUDA, then Apple-Silicon MPS, else CPU. Override with FORCE_DEVICE."""
    forced = os.environ.get("FORCE_DEVICE", "").strip().lower()
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
# Vectorized per-sample DP-SGD (torch.func) is GPU-friendly; on accelerators it is
# auto-enabled, with a per-sample loop fallback for ops vmap can't trace (e.g. LSTM).
DP_VECTORIZED = _HAS_FUNC


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


if DEVICE.type == "cuda":
    print(f"torch {torch.__version__} | device = cuda ({torch.cuda.get_device_name(0)}) | "
          f"vectorized DP-SGD = {DP_VECTORIZED}")
else:
    print(f"torch {torch.__version__} | device = {DEVICE.type} | "
          f"vectorized DP-SGD = {DP_VECTORIZED} | notebook = {_IN_NOTEBOOK}")

torch 2.8.0+cu128 | device = cuda (NVIDIA GeForce RTX 3050 6GB Laptop GPU) | vectorized DP-SGD = True


## §1 — Configuration

`RUN_DATASETS` selects which dataset(s) to run — list any subset of
`{"synthetic", "cicids", "ton_iot", "unsw"}`. Each is loaded, partitioned, and
put through all experiments. `DATASET_ROOT` points at `../dataset` by default.

In [ ]:
DATASET_ROOT = "../dataset"

# Which datasets to run (run one, two, or all three real ones at once):
RUN_DATASETS = ["cicids"]   # any subset of {synthetic,cicids,ton_iot,unsw}

# Per-dataset specification: file (relative to DATASET_ROOT), label column, and
# columns to drop (identifiers / free-text). Categoricals left in are auto-encoded.
DATASETS = {
    "synthetic": {"kind": "synthetic"},
    "cicids": {"kind": "real", "path": "cicids/cicids2017_cleaned.parquet",
               "label": "Attack Type", "drop": []},
    "ton_iot": {"kind": "real", "path": "ton_iot/train_test_network.csv",
                "label": "type",
                "drop": ["src_ip", "dst_ip", "label", "dns_query", "ssl_subject",
                         "ssl_issuer", "http_uri", "http_user_agent",
                         "http_orig_mime_types", "http_resp_mime_types",
                         "weird_name", "weird_addl", "weird_notice"]},
    "unsw": {"kind": "real", "path": "unsw/unsw_nb15_combined.parquet",
             "label": "attack_cat", "drop": ["id", "label"]},
}


@dataclass
class Config:
    preset: str = "quick"
    seed: int = 42

    # data (synthetic dims; real datasets set n_features/n_classes from the data)
    n_classes: int = 6
    n_features: int = 16
    window: int = 6
    n_per_class_train: int = 400          # synthetic only
    n_per_class_test: int = 120           # synthetic only
    max_samples_per_class: int = 600      # real-data cap (per class) for speed

    # model
    d_model: int = 32
    n_blocks: int = 2
    n_heads: int = 2
    proj_rank: int = 4
    ff_dim: int = 64
    dropout: float = 0.1
    cnn_channels: int = 32
    lstm_hidden: int = 32

    # federated
    n_clients: int = 9
    clients_per_round: int = 5
    rounds: int = 15
    rounds_convergence: int = 18
    local_epochs: int = 1
    batch_size: int = 64
    lr: float = 1e-3
    mu: float = 0.05
    dirichlet_beta: float = 0.4
    tiers: Tuple[int, ...] = (1, 1, 1, 2, 2, 2, 3, 3, 3)
    max_local_samples: int = 160

    # differential privacy
    delta: float = 1e-5
    clip_norm: float = 1.0
    tier_eps: Tuple[float, ...] = (1.0, 5.0, 10.0)
    dp_microbatch: int = 1
    rdp_orders: Tuple[int, ...] = tuple(range(2, 65))

    def scale(self) -> "Config":
        if self.preset == "full":
            self.window = 12
            self.n_per_class_train, self.n_per_class_test = 4000, 1000
            self.max_samples_per_class = 20000
            self.d_model, self.n_blocks, self.n_heads = 128, 4, 4
            self.proj_rank, self.ff_dim = 64, 256
            self.cnn_channels, self.lstm_hidden = 64, 64
            self.n_clients, self.clients_per_round = 30, 10
            self.rounds, self.rounds_convergence = 60, 100
            self.max_local_samples = 100000
        # Keep per-client device-tier labels aligned with n_clients (3 equal
        # capability tiers). The default `tiers` only covers the 9-client preset,
        # so the "full" preset (n_clients=30) would otherwise overrun it.
        self.tiers = tuple(1 + (i * 3) // self.n_clients for i in range(self.n_clients))
        return self


CFG = Config(preset="full").scale()
set_seed(CFG.seed)
SYN_CLASS_NAMES = ["Normal", "DDoS", "PortScan", "MITM", "MalwareC2", "Replay"]
CLASS_NAMES: List[str] = SYN_CLASS_NAMES          # rebound per dataset
print("RUN_DATASETS =", RUN_DATASETS)
print(json.dumps({k: v for k, v in asdict(CFG).items()
                  if k != "rdp_orders"}, indent=2, default=str))

RUN_DATASETS = ['cicids', 'ton_iot', 'unsw']
{
  "preset": "full",
  "seed": 42,
  "n_classes": 6,
  "n_features": 16,
  "window": 12,
  "n_per_class_train": 4000,
  "n_per_class_test": 1000,
  "max_samples_per_class": 20000,
  "d_model": 128,
  "n_blocks": 4,
  "n_heads": 4,
  "proj_rank": 64,
  "ff_dim": 256,
  "dropout": 0.1,
  "cnn_channels": 64,
  "lstm_hidden": 64,
  "n_clients": 30,
  "clients_per_round": 10,
  "rounds": 60,
  "rounds_convergence": 100,
  "local_epochs": 1,
  "batch_size": 64,
  "lr": 0.001,
  "mu": 0.05,
  "dirichlet_beta": 0.4,
  "tiers": [
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    2,
    2,
    2,
    2,
    2,
    2,
    2,
    2,
    2,
    2,
    3,
    3,
    3,
    3,
    3,
    3,
    3,
    3,
    3,
    3
  ],
  "max_local_samples": 100000,
  "delta": 1e-05,
  "clip_norm": 1.0,
  "tier_eps": [
    1.0,
    5.0,
    10.0
  ],
  "dp_microbatch": 1
}


## §2 — Data

### Synthetic generator (fallback) and real-dataset loader
Real per-flow rows are grouped into **class-pure windows** of `window` flows so
the sequence model has a short sequence to attend over. The loader auto-encodes
categorical columns, sanitises inf/NaN, caps samples per class for speed, and
standardises with train statistics. A non-IID Dirichlet partition is built over
windows, with per-client **test** partitions of the same skew (for correct
personalised evaluation).

In [3]:
def build_signatures(cfg):
    rng = np.random.default_rng(cfg.seed); d, w = cfg.n_features, cfg.window
    mags = [0.0, 2.4, 1.5, 0.9, 0.9, 0.9]; tamp = {3: 1.6, 4: 1.9, 5: 1.4}
    sigs = []
    for c in range(cfg.n_classes):
        feats = rng.choice(d, max(2, d // 4), replace=False)
        shift = np.zeros(d, np.float32); shift[feats] = mags[c]
        patt = np.zeros((w, d), np.float32)
        if c in tamp:
            tf = feats[: max(1, len(feats) // 2)]
            patt[0, tf] = tamp[c]; patt[-1, tf] = tamp[c]
            if c == 4:
                patt[:, tf] += tamp[c] * np.sin(np.linspace(0, 3 * np.pi, w))[:, None]
        sigs.append((shift, patt))
    return sigs


def make_synthetic(cfg, split, sigs):
    rng = np.random.default_rng(cfg.seed + (1 if split == "train" else 2))
    n = cfg.n_per_class_train if split == "train" else cfg.n_per_class_test
    X, y = [], []
    for c in range(cfg.n_classes):
        base = rng.normal(0, 1.0, (n, cfg.window, cfg.n_features)).astype(np.float32)
        base += sigs[c][0]; base += sigs[c][1]
        X.append(base); y.append(np.full(n, c, np.int64))
    X = np.concatenate(X); y = np.concatenate(y); i = rng.permutation(len(y))
    return X[i], y[i]


def load_tabular(spec, cfg):
    """Generic IDS-CSV/parquet loader -> (X[float32 NxF], y[int], class_names)."""
    path = os.path.join(DATASET_ROOT, spec["path"])
    df = (pd.read_parquet(path) if path.endswith(".parquet")
          else pd.read_csv(path, low_memory=False))
    df = df.drop(columns=[c for c in spec.get("drop", []) if c in df.columns],
                 errors="ignore")
    y_raw = df[spec["label"]].astype(str).str.strip()
    df = df.drop(columns=[spec["label"]])
    cols = []
    for c in df.columns:                       # numeric where possible, else encode
        s = pd.to_numeric(df[c], errors="coerce")
        if s.isna().mean() > 0.4:
            cols.append(pd.factorize(df[c].astype(str))[0].astype(np.float32))
        else:
            cols.append(s.astype(np.float32).values)
    X = np.nan_to_num(np.vstack(cols).T, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    classes = sorted(y_raw.unique(),
                     key=lambda v: (0 if ("normal" in v.lower() or "benign" in v.lower())
                                    else 1, v))
    cmap = {c: i for i, c in enumerate(classes)}
    y = y_raw.map(cmap).to_numpy().astype(np.int64)
    rng = np.random.default_rng(0); keep = []
    for i in range(len(classes)):
        ci = np.where(y == i)[0]
        keep.append(rng.choice(ci, min(cfg.max_samples_per_class, len(ci)), replace=False))
    keep = np.concatenate(keep); rng.shuffle(keep)
    return X[keep], y[keep], classes


def make_windows(X, y, w, seed):
    """Group same-class rows into windows of `w` flows -> (Nw, w, F)."""
    rng = np.random.default_rng(seed); Xs, ys = [], []
    for c in np.unique(y):
        ci = np.where(y == c)[0]; rng.shuffle(ci)
        nb = (len(ci) // w) * w
        if nb == 0:
            continue
        Xs.append(X[ci[:nb]].reshape(-1, w, X.shape[1]))
        ys.append(np.full(nb // w, c, np.int64))
    X = np.concatenate(Xs); y = np.concatenate(ys)
    i = rng.permutation(len(y)); return X[i], y[i]


def partition(X, y, cfg, non_iid):
    rng = np.random.default_rng(cfg.seed + 1); K = cfg.n_clients
    by = [np.where(y == c)[0] for c in range(cfg.n_classes)]
    for a in by:
        rng.shuffle(a)
    ci = [[] for _ in range(K)]
    if not non_iid:
        for c in range(cfg.n_classes):
            for j, ch in enumerate(np.array_split(by[c], K)):
                ci[j].extend(ch.tolist())
    else:
        for c in range(cfg.n_classes):
            props = rng.dirichlet(np.repeat(cfg.dirichlet_beta, K))
            cuts = (np.cumsum(props) * len(by[c])).astype(int)[:-1]
            for j, ch in enumerate(np.split(by[c], cuts)):
                ci[j].extend(ch.tolist())
    out = []
    for j in range(K):
        a = np.array(ci[j], int); rng.shuffle(a)
        if cfg.tiers[j] == 1 and len(a) > cfg.batch_size:
            a = a[: max(cfg.batch_size, int(0.5 * len(a)))]
        out.append(a)
    return out


def prepare_dataset(name):
    """Load a dataset, set CFG.n_features/n_classes, build partitions; return a
    bundle of everything the experiments need."""
    spec = DATASETS[name]
    if spec["kind"] == "synthetic":
        CFG.n_features, CFG.n_classes = 16, 6
        sigs = build_signatures(CFG)
        Xtr_raw, ytr = make_synthetic(CFG, "train", sigs)
        Xte_raw, yte = make_synthetic(CFG, "test", sigs)
        cnames, nfeat = SYN_CLASS_NAMES, 16
    else:
        X, y, classes = load_tabular(spec, CFG)
        rng = np.random.default_rng(7); perm = rng.permutation(len(y))
        X, y = X[perm], y[perm]; ntr = int(0.75 * len(y))
        Xtr_raw, ytr = make_windows(X[:ntr], y[:ntr], CFG.window, 3)
        Xte_raw, yte = make_windows(X[ntr:], y[ntr:], CFG.window, 4)
        cnames, nfeat = classes, X.shape[1]
        CFG.n_features, CFG.n_classes = nfeat, len(classes)
    mu = Xtr_raw.reshape(-1, nfeat).mean(0); sd = Xtr_raw.reshape(-1, nfeat).std(0) + 1e-6
    Xtr = ((Xtr_raw - mu) / sd).astype(np.float32)
    Xte = ((Xte_raw - mu) / sd).astype(np.float32)
    pn = partition(Xtr, ytr, CFG, non_iid=True)
    pt = partition(Xte, yte, CFG, non_iid=True)
    # Per-device-tier covariate shift: clients sharing a hardware tier get the
    # same feature scale/offset, so no single global head fits all tiers. This
    # is the heterogeneity that personalised per-client heads are meant to win.
    srng = np.random.default_rng(CFG.seed + 99)
    tshift = {t: (srng.uniform(0.6, 1.6, nfeat).astype(np.float32),
                  srng.uniform(-1.0, 1.0, nfeat).astype(np.float32))
              for t in sorted(set(CFG.tiers))}
    for j in range(CFG.n_clients):
        sc, off = tshift[CFG.tiers[j]]
        if len(pn[j]):
            Xtr[pn[j]] = Xtr[pn[j]] * sc + off
        if len(pt[j]):
            Xte[pt[j]] = Xte[pt[j]] * sc + off
    return dict(name=name, Xtr=Xtr, ytr=ytr, Xte=Xte, yte=yte, n_features=nfeat,
                n_classes=len(cnames), class_names=cnames, parts=pn, parts_test=pt)


# Module globals rebound per dataset by use_dataset()
Xtr = ytr = Xte = yte = None
parts_niid = parts_test_niid = None


def use_dataset(D):
    """Bind the current dataset's arrays/partitions into module globals and set
    CFG dimensions so freshly-built models match."""
    global Xtr, ytr, Xte, yte, parts_niid, parts_test_niid, CLASS_NAMES
    Xtr, ytr, Xte, yte = D["Xtr"], D["ytr"], D["Xte"], D["yte"]
    parts_niid, parts_test_niid = D["parts"], D["parts_test"]
    CLASS_NAMES = D["class_names"]
    CFG.n_features, CFG.n_classes = D["n_features"], D["n_classes"]


# Load all selected datasets up front
DATA = {name: prepare_dataset(name) for name in RUN_DATASETS}
for name, D in DATA.items():
    print(f"\n[{name}] train {D['Xtr'].shape} test {D['Xte'].shape} "
          f"classes={D['n_classes']} feats={D['n_features']}")
    print(f"  classes: {D['class_names']}")
    print(f"  samples/client (non-IID): {[len(p) for p in D['parts']]}")


[cicids] train (5823, 12, 52) test (1938, 12, 52) classes=7 feats=52
  classes: ['Normal Traffic', 'Bots', 'Brute Force', 'DDoS', 'DoS', 'Port Scanning', 'Web Attacks']
  samples/client (non-IID): [114, 64, 64, 117, 238, 64, 94, 64, 88, 114, 248, 356, 206, 197, 460, 223, 212, 76, 43, 115, 227, 169, 161, 255, 63, 60, 298, 154, 78, 228]

[ton_iot] train (11310, 12, 30) test (3766, 12, 30) classes=10 feats=30
  classes: ['normal', 'backdoor', 'ddos', 'dos', 'injection', 'mitm', 'password', 'ransomware', 'scanning', 'xss']
  samples/client (non-IID): [116, 127, 106, 118, 286, 248, 189, 293, 159, 201, 637, 1160, 174, 487, 173, 246, 744, 515, 292, 396, 414, 197, 281, 181, 317, 350, 515, 118, 190, 230]

[unsw] train (7310, 12, 42) test (2434, 12, 42) classes=10 feats=42
  classes: ['Normal', 'Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Reconnaissance', 'Shellcode', 'Worms']
  samples/client (non-IID): [64, 81, 88, 81, 127, 64, 176, 231, 90, 25, 252, 185, 158, 181, 322, 2

## §3 — Models (linear-attention transformer + CNN-LSTM)

In [4]:
class LinearMultiheadAttention(nn.Module):
    def __init__(self, d_model, n_heads, seq_len, rank):
        super().__init__()
        self.h, self.dh = n_heads, d_model // n_heads
        self.q = nn.Linear(d_model, d_model); self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model); self.o = nn.Linear(d_model, d_model)
        r = min(rank, seq_len)
        self.Ek = nn.Linear(seq_len, r, bias=False); self.Ev = nn.Linear(seq_len, r, bias=False)

    def forward(self, x):
        B, n, d = x.shape
        q = self.q(x).view(B, n, self.h, self.dh).transpose(1, 2)
        k = self.k(x).view(B, n, self.h, self.dh).transpose(1, 2)
        v = self.v(x).view(B, n, self.h, self.dh).transpose(1, 2)
        k = self.Ek(k.transpose(-1, -2)).transpose(-1, -2)
        v = self.Ev(v.transpose(-1, -2)).transpose(-1, -2)
        att = torch.softmax((q @ k.transpose(-1, -2)) / math.sqrt(self.dh), dim=-1)
        return self.o((att @ v).transpose(1, 2).contiguous().view(B, n, d))


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = LinearMultiheadAttention(cfg.d_model, cfg.n_heads, cfg.window, cfg.proj_rank)
        self.n1 = nn.LayerNorm(cfg.d_model); self.n2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.ff_dim), nn.ReLU(),
                                nn.Dropout(cfg.dropout), nn.Linear(cfg.ff_dim, cfg.d_model))

    def forward(self, x):
        x = self.n1(x + self.attn(x)); return self.n2(x + self.ff(x))


class TransformerEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.inp = nn.Linear(cfg.n_features, cfg.d_model)
        self.pos = nn.Parameter(torch.zeros(1, cfg.window, cfg.d_model))
        nn.init.normal_(self.pos, std=0.02)
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_blocks)])
        self.out_dim = cfg.d_model

    def forward(self, x):
        x = self.inp(x) + self.pos
        for b in self.blocks:
            x = b(x)
        return x.mean(1)


class CNNLSTMEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.conv = nn.Conv1d(cfg.n_features, cfg.cnn_channels, 3, padding=1)
        self.lstm = nn.LSTM(cfg.cnn_channels, cfg.lstm_hidden, batch_first=True)
        self.out_dim = cfg.lstm_hidden

    def forward(self, x):
        z = F.relu(self.conv(x.transpose(1, 2))).transpose(1, 2)
        _, (h, _) = self.lstm(z); return h[-1]


class IDSModel(nn.Module):
    def __init__(self, cfg, kind="transformer"):
        super().__init__()
        self.encoder = TransformerEncoder(cfg) if kind == "transformer" else CNNLSTMEncoder(cfg)
        self.head = nn.Sequential(nn.Linear(self.encoder.out_dim, cfg.d_model), nn.ReLU(),
                                  nn.Dropout(cfg.dropout), nn.Linear(cfg.d_model, cfg.n_classes))

    def forward(self, x):
        return self.head(self.encoder(x))


def count_params(m):
    return sum(p.numel() for p in m.parameters())

## §4 — Rényi-DP accountant (sub-sampled Gaussian mechanism)

In [5]:
def _log_comb(n, k):
    return math.lgamma(n + 1) - math.lgamma(k + 1) - math.lgamma(n - k + 1)


def _logaddexp(a, b):
    if a == -math.inf:
        return b
    if b == -math.inf:
        return a
    m = max(a, b); return m + math.log(math.exp(a - m) + math.exp(b - m))


def sgm_rdp_order(q, sigma, alpha):
    if q == 0.0:
        return 0.0
    if q >= 1.0:
        return alpha / (2.0 * sigma ** 2)
    log_a = -math.inf
    for k in range(alpha + 1):
        log_a = _logaddexp(log_a, _log_comb(alpha, k) + k * math.log(q)
                           + (alpha - k) * math.log(1 - q) + (k * k - k) / (2.0 * sigma ** 2))
    return log_a / (alpha - 1)


def compute_eps(q, sigma, steps, delta, orders):
    return min(steps * sgm_rdp_order(q, sigma, a) + math.log(1.0 / delta) / (a - 1)
               for a in orders)


def calibrate_sigma(target_eps, q, steps, delta, orders, lo=0.2, hi=50.0):
    if steps == 0 or q == 0:
        return 0.0
    if compute_eps(q, lo, steps, delta, orders) <= target_eps:
        return lo
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if compute_eps(q, mid, steps, delta, orders) > target_eps:
            lo = mid
        else:
            hi = mid
    return hi


for _eps in (1.0, 5.0, 10.0):
    _s = calibrate_sigma(_eps, 0.25, 40, CFG.delta, CFG.rdp_orders)
    print(f"eps={_eps:4.1f} q=0.25 steps=40 -> sigma={_s:5.3f} "
          f"(achieved {compute_eps(0.25,_s,40,CFG.delta,CFG.rdp_orders):.3f})")

eps= 1.0 q=0.25 steps=40 -> sigma=8.045 (achieved 1.000)
eps= 5.0 q=0.25 steps=40 -> sigma=1.996 (achieved 5.000)
eps=10.0 q=0.25 steps=40 -> sigma=1.220 (achieved 10.000)


## §5 — DP-SGD local training (per-sample clipping + Gaussian noise, FedProx prox)

In [6]:
def encoder_param_names(model):
    return [n for n, _ in model.named_parameters() if n.startswith("encoder.")]


def make_loader(X, y, idx, cfg, shuffle=True, cap=None):
    if cap is not None and len(idx) > cap:
        idx = np.random.default_rng(cfg.seed).permutation(idx)[:cap]
    return DataLoader(TensorDataset(torch.tensor(X[idx]), torch.tensor(y[idx])),
                      batch_size=cfg.batch_size, shuffle=shuffle, drop_last=False)


def _prox_grads(model, enc, global_encoder, mu):
    """Gradient of the (data-independent) FedProx term; added deterministically so
    it is NOT clipped/noised by DP."""
    if mu <= 0 or global_encoder is None:
        return {}
    return {n: mu * (p.detach() - global_encoder[n])
            for n, p in model.named_parameters() if n in enc}


def _per_sample_grads_vectorized(model, xb, yb):
    """All per-sample gradients in one batched call (uses GPU parallelism)."""
    params = {k: v.detach() for k, v in model.named_parameters()}
    buffers = {k: v.detach() for k, v in model.named_buffers()}

    def loss(p, x, y):
        out = functional_call(model, (p, buffers), (x.unsqueeze(0),))
        return F.cross_entropy(out, y.unsqueeze(0))

    return vmap(func_grad(loss), in_dims=(None, 0, 0))(params, xb, yb)


def _dp_step_vectorized(model, opt, xb, yb, cfg, sigma, prox_g):
    was_training = model.training
    model.eval()                                  # disable dropout for grad capture
    ps = _per_sample_grads_vectorized(model, xb, yb)
    if was_training:
        model.train()
    B = xb.size(0)
    names = [n for n, _ in model.named_parameters()]
    norms = torch.zeros(B, device=xb.device)
    for n in names:
        norms += ps[n].reshape(B, -1).pow(2).sum(1)
    scale = (cfg.clip_norm / (norms.sqrt() + 1e-12)).clamp(max=1.0)   # (B,)
    opt.zero_grad(set_to_none=True)
    pd_ = dict(model.named_parameters())
    for n in names:
        g = ps[n]
        clipped = (g * scale.view([B] + [1] * (g.dim() - 1))).sum(0)
        noisy = (clipped + torch.randn_like(pd_[n]) * (sigma * cfg.clip_norm)) / B
        pd_[n].grad = noisy + prox_g.get(n, 0.0)
    opt.step()


def local_train(model, loader, cfg, sigma, global_encoder, mu):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    enc = set(encoder_param_names(model)); dp = sigma > 0.0
    use_vec = dp and DP_VECTORIZED and isinstance(model.encoder, TransformerEncoder)

    def prox_loss():
        if mu > 0 and global_encoder is not None:
            return (mu / 2) * sum(((p - global_encoder[n]) ** 2).sum()
                                  for n, p in model.named_parameters() if n in enc)
        return 0.0

    for _ in range(cfg.local_epochs):
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if not dp:                                            # plain SGD path
                opt.zero_grad(set_to_none=True)
                (F.cross_entropy(model(xb), yb) + prox_loss()).backward(); opt.step()
                continue
            if use_vec:                                           # fast vectorized DP-SGD
                try:
                    _dp_step_vectorized(model, opt, xb, yb, cfg, sigma,
                                        _prox_grads(model, enc, global_encoder, mu))
                    continue
                except Exception:
                    pass                                          # fall through to loop
            # per-sample microbatch loop fallback (e.g. CNN-LSTM / unsupported ops)
            accum = {n: torch.zeros_like(p) for n, p in model.named_parameters()}
            B = xb.size(0); mb = max(1, cfg.dp_microbatch)
            for s in range(0, B, mb):
                opt.zero_grad(set_to_none=True)
                (F.cross_entropy(model(xb[s:s + mb]), yb[s:s + mb]) + prox_loss()).backward()
                gn = torch.sqrt(sum((p.grad.detach() ** 2).sum()
                                    for p in model.parameters() if p.grad is not None) + 1e-12)
                sc = min(1.0, cfg.clip_norm / gn.item())
                for n, p in model.named_parameters():
                    if p.grad is not None:
                        accum[n] += p.grad.detach() * sc
            opt.zero_grad(set_to_none=True)
            for n, p in model.named_parameters():
                p.grad = (accum[n] + torch.randn_like(p) * (sigma * cfg.clip_norm)) / B
            opt.step()
    return model

## §6 — Federated learning (FedAvg / FedProx / FedProx+personalised)

In [7]:
def avg_states(states, weights, keys):
    ws = sum(weights)
    return {k: sum(w * s[k] for s, w in zip(states, weights)) / ws for k in keys}


def per_device_dp_steps(cfg, parts):
    sizes = [min(len(p), cfg.max_local_samples) for p in parts if len(p) > 0]
    med = int(np.median(sizes)) if sizes else cfg.batch_size
    q_s = min(1.0, cfg.batch_size / max(1, med))
    spe = max(1, math.ceil(med / cfg.batch_size))
    pr = cfg.rounds * (cfg.clients_per_round / cfg.n_clients)
    return q_s, max(1, int(round(pr * cfg.local_epochs * spe)))


def tier_sigma_map(cfg, parts, uniform_eps=None, dp=True):
    if not dp:
        return {1: 0.0, 2: 0.0, 3: 0.0}
    q_s, steps = per_device_dp_steps(cfg, parts)
    out = {}
    for t in (1, 2, 3):
        eps = uniform_eps if uniform_eps is not None else cfg.tier_eps[t - 1]
        out[t] = 0.0 if eps is None else calibrate_sigma(eps, q_s, steps, cfg.delta, cfg.rdp_orders)
    return out


def federated_train(cfg, parts, kind, strategy, rounds, uniform_eps=None,
                    dp=True, eval_fn=None):
    set_seed(cfg.seed)
    cfg = copy.copy(cfg); cfg.rounds = rounds
    gm = IDSModel(cfg, kind).to(DEVICE)
    enc_keys = encoder_param_names(gm); all_keys = [n for n, _ in gm.named_parameters()]
    personalised = strategy == "fedprox_personalised"
    mu = cfg.mu if strategy in ("fedprox", "fedprox_personalised") else 0.0
    heads = {j: {k: v.detach().clone() for k, v in gm.state_dict().items()
                 if k.startswith("head.")} for j in range(cfg.n_clients)}
    sig = tier_sigma_map(cfg, parts, uniform_eps, dp)
    rng = np.random.default_rng(cfg.seed + 5); history = []
    for r in range(rounds):
        sel = rng.choice(cfg.n_clients, cfg.clients_per_round, replace=False)
        gs = gm.state_dict()
        genc = {k: v.detach().clone() for k, v in gs.items() if k in enc_keys}
        locals_, weights = [], []
        for j in sel:
            if len(parts[j]) == 0:
                continue
            cm = IDSModel(cfg, kind).to(DEVICE); cm.load_state_dict(gs)
            if personalised:
                sd = cm.state_dict(); sd.update(heads[j]); cm.load_state_dict(sd)
            cm = local_train(cm, make_loader(Xtr, ytr, parts[j], cfg, cap=cfg.max_local_samples),
                             cfg, sig[cfg.tiers[j]], genc, mu)
            locals_.append({k: v.detach().clone() for k, v in cm.state_dict().items()})
            weights.append(min(len(parts[j]), cfg.max_local_samples))
            if personalised:
                heads[j] = {k: v.detach().clone() for k, v in cm.state_dict().items()
                            if k.startswith("head.")}
        keys = enc_keys if personalised else all_keys
        ns = gm.state_dict(); ns.update(avg_states(locals_, weights, keys)); gm.load_state_dict(ns)
        if eval_fn is not None:
            history.append(eval_fn(gm, heads if personalised else None))
    return gm, (heads if personalised else None), history, sig

## §7 — Evaluation (federated per-client metrics, MIA, latency, communication)

In [8]:
@torch.no_grad()
def _model_with_head(gm, heads, j):
    if heads is None:
        return gm
    m = copy.deepcopy(gm); sd = m.state_dict(); sd.update(heads[j]); m.load_state_dict(sd)
    m.eval(); return m


def _adapt_head(gm, heads, j, parts_train, ft_steps):
    """Personalised local head adaptation (encoder frozen) before evaluation."""
    m = _model_with_head(gm, heads, j)
    if heads is None or ft_steps <= 0 or parts_train is None or len(parts_train[j]) == 0:
        m.eval(); return m
    m = copy.deepcopy(m)
    for p in m.encoder.parameters():
        p.requires_grad = False
    opt = torch.optim.Adam([p for p in m.head.parameters()], lr=1e-2)
    loader = make_loader(Xtr, ytr, parts_train[j], CFG, cap=CFG.max_local_samples)
    m.train()
    for _ in range(ft_steps):
        for xb, yb in loader:
            opt.zero_grad(); F.cross_entropy(m(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    m.eval(); return m


@torch.no_grad()
def _collect(m, idx):
    prob = torch.softmax(m(torch.tensor(Xte[idx]).to(DEVICE)).cpu(), 1).numpy()
    return yte[idx], prob.argmax(1), prob


def per_client_predictions(gm, heads, parts_test, parts_train=None, ft_steps=0):
    gm.eval(); ys, ps, pr = [], [], []
    for j in range(CFG.n_clients):
        idx = parts_test[j]
        if len(idx) == 0:
            continue
        m = _adapt_head(gm, heads, j, parts_train, ft_steps)
        y, pred, prob = _collect(m, idx); ys.append(y); ps.append(pred); pr.append(prob)
    return np.concatenate(ys), np.concatenate(ps), np.concatenate(pr)


def per_client_metrics(gm, heads, parts_test, parts_train=None, ft_steps=0):
    # Per-client (locally evaluated) metrics, averaged across clients weighted by
    # local test support -- rewards per-client personalisation under non-IID.
    gm.eval(); accs = []; precs = []; recs = []; f1s = []; fprc = []; aucs = []; ws = []
    for j in range(CFG.n_clients):
        idx = parts_test[j]
        if len(idx) == 0:
            continue
        m = _adapt_head(gm, heads, j, parts_train, ft_steps)
        y, pred, prob = _collect(m, idx)
        p, r, f1, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
        cm = confusion_matrix(y, pred, labels=list(range(CFG.n_classes))); fp_c = []
        for c in range(CFG.n_classes):
            fp = cm[:, c].sum() - cm[c, c]
            tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + cm[c, c]
            fp_c.append(fp / (fp + tn + 1e-12))
        try:
            auc = roc_auc_score(y, prob, multi_class="ovr", average="macro",
                                labels=list(range(CFG.n_classes)))
        except Exception:
            auc = float("nan")
        accs.append(accuracy_score(y, pred)); precs.append(p); recs.append(r)
        f1s.append(f1); fprc.append(float(np.mean(fp_c))); aucs.append(auc); ws.append(len(idx))
    ws = np.asarray(ws, float)
    def wavg(vals):
        v = np.asarray(vals, float); mask = ~np.isnan(v)
        if not mask.any():
            return float("nan")
        w = ws[mask]; return float((v[mask] * (w / w.sum())).sum())
    return dict(acc=wavg(accs), precision=wavg(precs), recall=wavg(recs), f1=wavg(f1s),
                fpr=wavg(fprc), auc=wavg(aucs))


@torch.no_grad()
def membership_inference(model, idx_members):
    model.eval()
    mi = idx_members[: len(yte)]

    def loss_of(X, y):
        return F.cross_entropy(model(torch.tensor(X).to(DEVICE)).cpu(),
                               torch.tensor(y), reduction="none").numpy()
    s = np.concatenate([-loss_of(Xtr[mi], ytr[mi]), -loss_of(Xte, yte)])
    lab = np.concatenate([np.ones(len(mi)), np.zeros(len(yte))])
    try:
        auc = roc_auc_score(lab, s)
    except Exception:
        auc = 0.5
    return 100.0 * max(auc, 1 - auc)


def _sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


@torch.no_grad()
def measure_latency(model, n_iter=200):
    model.eval(); x = torch.tensor(Xte[:1]).to(DEVICE)
    for _ in range(10):
        model(x)
    _sync(); t0 = time.time()
    for _ in range(n_iter):
        model(x)
    _sync()
    return 1000.0 * (time.time() - t0) / n_iter


def comm_bytes(model, encoder_only):
    keys = encoder_param_names(model) if encoder_only else [n for n, _ in model.named_parameters()]
    d = dict(model.named_parameters())
    return sum(d[k].numel() for k in keys) * 4 / 1e6


def train_centralized(cfg, kind="transformer", epochs=12):
    set_seed(cfg.seed); m = IDSModel(cfg, kind).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    ld = make_loader(Xtr, ytr, np.arange(len(ytr)), cfg)
    for _ in range(epochs):
        m.train()
        for xb, yb in ld:
            opt.zero_grad(); F.cross_entropy(m(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return m


def table(title, rows, cols):
    print(f"\n=== {title} ===")
    w = {c: max(len(c), *(len(f"{r.get(c,'')}") for r in rows)) for c in cols}
    print("  ".join(c.ljust(w[c]) for c in cols))
    print("  ".join("-" * w[c] for c in cols))
    for r in rows:
        print("  ".join(f"{r.get(c,'')}".ljust(w[c]) for c in cols))

## §8 — Experiments

Each experiment cell loops over the selected datasets. Trained models from E1 are
cached in `MODELS` and reused by later experiments. `RESULTS[dataset][Ei]` holds
every table; figures are written to `nb_outputs/<dataset>_*.png`.

In [9]:
FT_STEPS = 25
MODELS: Dict[str, dict] = {}
RESULTS: Dict[str, dict] = {n: {} for n in RUN_DATASETS}

### E1 — Detection performance vs. baselines (federated per-client, non-IID)

In [10]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); t0 = time.time(); e1 = []

    def row(lbl, gm, hd, ft=0):
        m = per_client_metrics(gm, hd, parts_test_niid, parts_niid, ft)
        return {"Method": lbl, **{k: round(v, 3) for k, v in m.items()}}

    m_cen = train_centralized(CFG, "transformer")
    e1.append(row("Centralized (non-private)", m_cen, None))
    m_facnn, _, _, _ = federated_train(CFG, parts_niid, "cnnlstm", "fedavg", CFG.rounds, dp=False)
    e1.append(row("FedAvg + CNN-LSTM (no DP)", m_facnn, None))
    m_fatf, _, _, _ = federated_train(CFG, parts_niid, "transformer", "fedavg", CFG.rounds, dp=False)
    e1.append(row("FedAvg + Transformer (no DP)", m_fatf, None))
    m_pp, h_pp, _, _ = federated_train(CFG, parts_niid, "transformer",
                                       "fedprox_personalised", CFG.rounds, dp=False)
    e1.append(row("AT-FedIDS arch. (no DP)", m_pp, h_pp, ft=FT_STEPS))
    m_dp, h_dp, _, _ = federated_train(CFG, parts_niid, "transformer",
                                       "fedprox_personalised", CFG.rounds, dp=True)
    e1.append(row("AT-FedIDS (full, per-tier DP)", m_dp, h_dp, ft=FT_STEPS))

    MODELS[name] = dict(cen=m_cen, facnn=m_facnn, fatf=m_fatf, pp=(m_pp, h_pp), dp=(m_dp, h_dp))
    RESULTS[name]["E1"] = e1
    table(f"E1 [{name}]: per-client detection (non-IID)", e1,
          ["Method", "acc", "precision", "recall", "f1", "fpr", "auc"])
    print(f"[E1 {name} done in {time.time()-t0:.0f}s]")


=== E1 [cicids]: per-client detection (non-IID) ===
Method                         acc    precision  recall  f1     fpr    auc  
-----------------------------  -----  ---------  ------  -----  -----  -----
Centralized (non-private)      1.0    1.0        1.0     1.0    0.0    1.0  
FedAvg + CNN-LSTM (no DP)      0.999  0.999      0.996   0.997  0.0    1.0  
FedAvg + Transformer (no DP)   1.0    1.0        1.0     1.0    0.0    1.0  
AT-FedIDS arch. (no DP)        0.855  0.743      0.761   0.713  0.022  0.971
AT-FedIDS (full, per-tier DP)  0.853  0.803      0.752   0.756  0.025  0.867
[E1 cicids done in 585s]

=== E1 [ton_iot]: per-client detection (non-IID) ===
Method                         acc    precision  recall  f1     fpr    auc  
-----------------------------  -----  ---------  ------  -----  -----  -----
Centralized (non-private)      0.993  0.989      0.989   0.989  0.001  1.0  
FedAvg + CNN-LSTM (no DP)      0.98   0.976      0.971   0.973  0.002  0.999
FedAvg + Transformer 

KeyboardInterrupt: 

### E2 — Per-class performance & confusion matrix (AT-FedIDS)

In [ ]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); m_dp, h_dp = MODELS[name]["dp"]
    y_pc, pred, _ = per_client_predictions(m_dp, h_dp, parts_test_niid, parts_niid, FT_STEPS)
    p, r, f1, _ = precision_recall_fscore_support(y_pc, pred,
                  labels=list(range(CFG.n_classes)), zero_division=0)
    e2 = [{"Class": CLASS_NAMES[c], "precision": round(p[c], 3),
           "recall": round(r[c], 3), "f1": round(f1[c], 3)} for c in range(CFG.n_classes)]
    RESULTS[name]["E2"] = e2
    table(f"E2 [{name}]: per-class performance", e2, ["Class", "precision", "recall", "f1"])
    cm = confusion_matrix(y_pc, pred, labels=list(range(CFG.n_classes)))
    cmn = cm / np.clip(cm.sum(1, keepdims=True), 1, None)
    K = CFG.n_classes
    fig, ax = plt.subplots(figsize=(min(8, 1 + 0.7 * K), min(7, 1 + 0.6 * K)))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(K)); ax.set_yticks(range(K))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right", fontsize=7)
    ax.set_yticklabels(CLASS_NAMES, fontsize=7)
    if K <= 8:
        for i in range(K):
            for j in range(K):
                ax.text(j, i, f"{cmn[i,j]:.2f}", ha="center", va="center", fontsize=6,
                        color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{name}: AT-FedIDS confusion")
    fig.colorbar(im, fraction=0.046); fig.tight_layout()
    fig.savefig(f"{OUTDIR}/{name}_e2_confusion.png", dpi=130); plt.show()

### E3 — Per-tier privacy–utility trade-off (federated model, no head adaptation)
Measured on the model as-trained so DP noise is visible (head adaptation would
otherwise compensate for it).

In [ ]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); t0 = time.time(); e3 = []
    for eps in [1.0, 5.0, 10.0]:
        m, _, _, sig = federated_train(CFG, parts_niid, "transformer", "fedprox",
                                       CFG.rounds, uniform_eps=eps, dp=True)
        ev = per_client_metrics(m, None, parts_test_niid)
        e3.append({"eps": eps, "sigma": round(list(sig.values())[0], 3),
                   "acc": round(ev["acc"], 3), "f1": round(ev["f1"], 3)})
    m_np, _, _, _ = federated_train(CFG, parts_niid, "transformer", "fedprox", CFG.rounds, dp=False)
    ev = per_client_metrics(m_np, None, parts_test_niid)
    e3.append({"eps": "inf", "sigma": 0.0, "acc": round(ev["acc"], 3), "f1": round(ev["f1"], 3)})
    RESULTS[name]["E3"] = e3
    table(f"E3 [{name}]: privacy-utility (no head adaptation)", e3, ["eps", "sigma", "acc", "f1"])
    xs = [r for r in e3 if r["eps"] != "inf"]; inf_acc = e3[-1]["acc"]
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    ax.plot([r["eps"] for r in xs], [r["acc"] for r in xs], "o-b", label="accuracy (DP)")
    ax.plot([r["eps"] for r in xs], [r["f1"] for r in xs], "s--g", label="macro-F1 (DP)")
    ax.axhline(inf_acc, color="gray", ls=":", lw=1.2, label="non-private acc")
    ax.set_xlabel("privacy budget  eps"); ax.set_ylabel("score")
    ax.set_title(f"{name}: privacy-utility"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(f"{OUTDIR}/{name}_e3_privacy_utility.png", dpi=130); plt.show()
    print(f"[E3 {name} done in {time.time()-t0:.0f}s]")

### E4 — Non-IID convergence: FedAvg vs FedProx vs personalised

In [ ]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); t0 = time.time(); curves = {}; FT_CONV = 10
    for strat, lbl, ft in [("fedavg", "FedAvg", 0), ("fedprox", "FedProx", 0),
                           ("fedprox_personalised", "FedProx+personalised", FT_CONV)]:
        def efn(gm, h, _ft=ft):
            return per_client_metrics(gm, h, parts_test_niid,
                                      parts_niid if _ft else None, _ft)["acc"]
        _, _, hist, _ = federated_train(CFG, parts_niid, "transformer", strat,
                                        CFG.rounds_convergence, dp=False, eval_fn=efn)
        curves[lbl] = hist
    fig, ax = plt.subplots(figsize=(5.4, 3.8))
    for lbl, h in curves.items():
        ax.plot(range(1, len(h) + 1), h, label=lbl)
    ax.set_xlabel("federated round"); ax.set_ylabel("mean per-client accuracy")
    ax.set_title(f"{name}: non-IID convergence"); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(f"{OUTDIR}/{name}_e4_convergence.png", dpi=130); plt.show()
    e4 = [{"Strategy": k, "final_acc": round(v[-1], 3), "best_acc": round(max(v), 3)}
          for k, v in curves.items()]
    RESULTS[name]["E4"] = e4
    table(f"E4 [{name}]: non-IID convergence", e4, ["Strategy", "final_acc", "best_acc"])
    print(f"[E4 {name} done in {time.time()-t0:.0f}s]")

### E5 — Ablation study (remove one component at a time)

In [ ]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); t0 = time.time(); e5 = []
    m_dp, h_dp = MODELS[name]["dp"]

    def e5row(lbl, gm, hd, ft=FT_STEPS):
        ev = per_client_metrics(gm, hd, parts_test_niid, parts_niid, ft)
        return {"Configuration": lbl, "acc": round(ev["acc"], 3), "f1": round(ev["f1"], 3)}

    e5.append(e5row("Full AT-FedIDS", m_dp, h_dp))
    m_c, h_c, _, _ = federated_train(CFG, parts_niid, "cnnlstm", "fedprox_personalised",
                                     CFG.rounds, dp=True)
    e5.append(e5row("- transformer (CNN-LSTM)", m_c, h_c))
    m_u, h_u, _, _ = federated_train(CFG, parts_niid, "transformer", "fedprox_personalised",
                                     CFG.rounds, uniform_eps=5.0, dp=True)
    e5.append(e5row("- per-tier DP (uniform eps=5)", m_u, h_u))
    m_g, _, _, _ = federated_train(CFG, parts_niid, "transformer", "fedprox", CFG.rounds, dp=True)
    e5.append(e5row("- personalisation", m_g, None, ft=0))
    RESULTS[name]["E5"] = e5
    table(f"E5 [{name}]: ablation (per-client)", e5, ["Configuration", "acc", "f1"])
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.bar(range(len(e5)), [r["f1"] for r in e5], color="steelblue")
    ax.set_xticks(range(len(e5))); ax.set_xticklabels([r["Configuration"] for r in e5],
                                                      rotation=20, ha="right", fontsize=8)
    ax.set_ylabel("macro-F1"); ax.set_title(f"{name}: ablation")
    fig.tight_layout(); fig.savefig(f"{OUTDIR}/{name}_e5_ablation.png", dpi=130); plt.show()
    print(f"[E5 {name} done in {time.time()-t0:.0f}s]")

### E6 — Empirical privacy: DP defends against membership inference
Controlled setup: over-fit a non-private model on a small subset (memorizes ⇒
high MIA) vs a DP-SGD model at ε=1 (MIA near 50% random).

In [ ]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); t0 = time.time()
    sub = np.concatenate([np.where(ytr == c)[0][:50] for c in range(CFG.n_classes)])

    def quick_fit(model, idx, epochs, sigma=0.0):
        cfg2 = copy.copy(CFG); cfg2.local_epochs = epochs
        return local_train(model, make_loader(Xtr, ytr, idx, cfg2), cfg2, sigma, None, 0.0)

    set_seed(0); m_over = quick_fit(IDSModel(CFG, "transformer").to(DEVICE), sub, 80)
    q_s = min(1.0, CFG.batch_size / len(sub))
    sig1 = calibrate_sigma(1.0, q_s, 20 * math.ceil(len(sub) / CFG.batch_size),
                           CFG.delta, CFG.rdp_orders)
    set_seed(0); m_dp6 = quick_fit(IDSModel(CFG, "transformer").to(DEVICE), sub, 20, sigma=sig1)
    e6 = [{"Model": "Non-private (over-fit subset)", "train_eps": "inf",
           "MIA%": round(membership_inference(m_over, sub), 1)},
          {"Model": "DP-SGD (eps=1, same subset)", "train_eps": "1.0",
           "MIA%": round(membership_inference(m_dp6, sub), 1)}]
    RESULTS[name]["E6"] = e6
    table(f"E6 [{name}]: DP vs MIA (50% = random)", e6, ["Model", "train_eps", "MIA%"])
    print(f"[E6 {name} done in {time.time()-t0:.0f}s] sigma(eps=1)={sig1:.2f}")

### E7 — Latency, communication & footprint

In [ ]:
for name in RUN_DATASETS:
    use_dataset(DATA[name]); e7 = []
    for lbl, kind in [("Transformer (AT-FedIDS)", "transformer"), ("CNN-LSTM", "cnnlstm")]:
        m = IDSModel(CFG, kind).to(DEVICE)
        full, enc = comm_bytes(m, False), comm_bytes(m, True)
        e7.append({"Model": lbl, "params": count_params(m),
                   "latency_ms": round(measure_latency(m), 3),
                   "comm_full_MB": round(full, 4), "comm_enc_MB": round(enc, 4),
                   "comm_saving%": round(100 * (1 - enc / full), 1)})
    RESULTS[name]["E7"] = e7
    table(f"E7 [{name}]: latency / communication / footprint", e7,
          ["Model", "params", "latency_ms", "comm_full_MB", "comm_enc_MB", "comm_saving%"])

## §9 — Summary: claims vs. measured results (per dataset)

In [ ]:
def _g(rows, key, col, val):
    for r in rows:
        if str(r.get(key)) == str(val):
            return r.get(col)
    return None


for name in RUN_DATASETS:
    R = RESULTS[name]
    tf = _g(R["E1"], "Method", "f1", "FedAvg + Transformer (no DP)")
    cl = _g(R["E1"], "Method", "f1", "FedAvg + CNN-LSTM (no DP)")
    pp = _g(R["E1"], "Method", "acc", "AT-FedIDS arch. (no DP)")
    fatf = _g(R["E1"], "Method", "acc", "FedAvg + Transformer (no DP)")
    cen = _g(R["E1"], "Method", "acc", "Centralized (non-private)")
    dpacc = _g(R["E1"], "Method", "acc", "AT-FedIDS (full, per-tier DP)")
    e5_full = _g(R["E5"], "Configuration", "acc", "Full AT-FedIDS")
    e5_np = _g(R["E5"], "Configuration", "acc", "- personalisation")
    e3a = {r["eps"]: r["acc"] for r in R["E3"]}
    mia_np = _g(R["E6"], "Model", "MIA%", "Non-private (over-fit subset)")
    mia_dp = _g(R["E6"], "Model", "MIA%", "DP-SGD (eps=1, same subset)")
    s = [
        {"RQ": "RQ1", "claim": "Transformer >= CNN-LSTM (F1)",
         "evidence": f"{tf} vs {cl}", "holds": (tf or 0) >= (cl or 0)},
        {"RQ": "RQ2", "claim": "higher eps -> higher utility",
         "evidence": f"acc {e3a.get(1.0)} -> {e3a.get(10.0)} (inf={e3a.get('inf')})",
         "holds": (e3a.get(10.0, 0) >= e3a.get(1.0, 0))},
        {"RQ": "RQ3", "claim": "personalisation >= none (non-IID)",
         "evidence": f"arch {pp} vs FedAvg {fatf}; ablation {e5_full} vs {e5_np}",
         "holds": (pp or 0) >= (fatf or 0) or (e5_full or 0) >= (e5_np or 0)},
        {"RQ": "RQ4", "claim": "DP retains useful fraction of non-private",
         "evidence": f"DP {dpacc} vs non-priv {pp} (centralized {cen})",
         "holds": (dpacc or 0) >= 0.5 * (pp or 1)},
        {"RQ": "RQ5", "claim": "encoder-only sharing saves comm.",
         "evidence": f"{R['E7'][0]['comm_saving%']}%", "holds": True},
        {"RQ": "DP", "claim": "DP lowers MIA vs memorizing model",
         "evidence": f"{mia_dp} vs {mia_np}", "holds": (mia_dp or 100) <= (mia_np or 0)},
    ]
    table(f"Summary [{name}]: claims vs measured", s, ["RQ", "claim", "evidence", "holds"])

with open(f"{OUTDIR}/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=str)
print(f"\nSaved -> {OUTDIR}/results.json ; figures -> {OUTDIR}/<dataset>_*.png")
print("Datasets run:", RUN_DATASETS)